# ARC-AGI-3 Submission — Adaptive Reasoning Agent

This notebook runs the **AdaptiveReasoning** agent (v4_1 architecture) against
all ARC-AGI-3 games in offline mode for Kaggle submission.

**Architecture:** Grid analysis → Game memory → Strategy routing (wall-following + goal-seeking)

No LLM or internet required — pure algorithmic reasoning.

In [ ]:
import subprocess, sys, os

# Install dependencies (Kaggle environment)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'arc-agi>=0.9', 'arcengine>=0.9', 'numpy', 'pydantic'])
print('Dependencies installed.')

In [ ]:
# Configure offline mode
os.environ['OPERATION_MODE'] = 'offline'
os.environ.setdefault('ARC_API_KEY', 'kaggle')
os.environ.setdefault('RECORDINGS_DIR', 'recordings')

# Kaggle provides environment files at /kaggle/input/
# Adjust this path based on your Kaggle dataset mount
KAGGLE_ENV_DIR = '/kaggle/input/arc-agi-3/environment_files'
LOCAL_ENV_DIR = '../environment_files'

if os.path.exists(KAGGLE_ENV_DIR):
    ENVIRONMENTS_DIR = KAGGLE_ENV_DIR
    print(f'Running on Kaggle: {KAGGLE_ENV_DIR}')
elif os.path.exists(LOCAL_ENV_DIR):
    ENVIRONMENTS_DIR = LOCAL_ENV_DIR
    print(f'Running locally: {LOCAL_ENV_DIR}')
else:
    raise RuntimeError('No environment_files directory found!')

os.environ['ENVIRONMENTS_DIR'] = ENVIRONMENTS_DIR

In [ ]:
# Add project paths
import pathlib

NOTEBOOK_DIR = pathlib.Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AGENTS_DIR = PROJECT_ROOT / 'ARC-AGI-3-Agents'

for p in [str(PROJECT_ROOT), str(AGENTS_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Project root: {PROJECT_ROOT}')
print(f'Agents dir:   {AGENTS_DIR}')

In [ ]:
# Verify imports
from v4_1_reasoning_system.arc_agi.grid_analyzer import GridAnalyzer
from v4_1_reasoning_system.arc_agi.game_memory import GameMemory
from v4_1_reasoning_system.arc_agi.strategy_router import StrategyRouter
print('v4_1 arc_agi package: OK')

from arc_agi import Arcade, OperationMode
from arcengine import FrameData, GameAction, GameState
print('arcengine: OK')

In [ ]:
# Discover games
arc = Arcade(
    operation_mode=OperationMode.OFFLINE,
    environments_dir=ENVIRONMENTS_DIR,
)
envs = arc.get_environments()
game_ids = [e.game_id for e in envs]
print(f'Found {len(game_ids)} games')
print('First 5:', game_ids[:5])

In [ ]:
import logging, time, json
from typing import Dict, Any, List

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
)
logger = logging.getLogger('submission')

In [ ]:
# Import the agent (after path setup)
os.chdir(str(AGENTS_DIR))  # agent expects to be in ARC-AGI-3-Agents/
from agents.templates.adaptive_reasoning_agent import AdaptiveReasoning

# Configuration
MAX_ACTIONS_PER_GAME = 300   # Balance exploration vs time budget
MAX_TOTAL_SECONDS = 5 * 3600 # 5 hours (leave 1hr buffer from 6hr limit)

AdaptiveReasoning.MAX_ACTIONS = MAX_ACTIONS_PER_GAME
print(f'Agent: AdaptiveReasoning, max_actions={MAX_ACTIONS_PER_GAME}')

In [ ]:
def run_single_game(arc_instance, game_id: str, max_actions: int) -> Dict[str, Any]:
    """Run the agent on a single game and return results."""
    start = time.time()
    result = {
        'game_id': game_id,
        'status': 'error',
        'actions': 0,
        'max_level': 0,
        'time': 0.0,
    }

    try:
        env = arc_instance.make(game_id)
        agent = AdaptiveReasoning(
            card_id=None,
            game_id=game_id,
            agent_name='adaptivereasoning',
            ROOT_URL='',
            record=False,
            arc_env=env,
        )
        agent.main()

        summary = agent.memory.summary()
        elapsed = time.time() - start

        result.update({
            'status': 'win' if agent.state == GameState.WIN else 'done',
            'actions': agent.action_counter,
            'max_level': summary['max_level'],
            'states_visited': summary['states_visited'],
            'exploration': round(summary['exploration_score'], 3),
            'player_found': summary['player_identified'],
            'semantics': summary['action_semantics'],
            'time': round(elapsed, 2),
        })
    except Exception as e:
        result['error'] = str(e)
        result['time'] = round(time.time() - start, 2)
        logger.error(f'Game {game_id} failed: {e}')

    return result

In [ ]:
# Run all games
results: List[Dict[str, Any]] = []
global_start = time.time()

for i, gid in enumerate(game_ids):
    elapsed_total = time.time() - global_start
    if elapsed_total > MAX_TOTAL_SECONDS:
        logger.warning(f'Time budget exhausted after {i} games ({elapsed_total:.0f}s)')
        break

    logger.info(f'[{i+1}/{len(game_ids)}] Starting game: {gid}')
    r = run_single_game(arc, gid, MAX_ACTIONS_PER_GAME)
    results.append(r)

    status_icon = '\u2705' if r['status'] == 'win' else '\u274c' if r['status'] == 'error' else '\u2B1C'
    print(f"  {status_icon} {gid}: lvl={r['max_level']}, acts={r['actions']}, {r['time']}s")

total_time = time.time() - global_start
print(f'\nCompleted {len(results)}/{len(game_ids)} games in {total_time:.1f}s')

In [ ]:
# Summary
wins = sum(1 for r in results if r['status'] == 'win')
errors = sum(1 for r in results if r['status'] == 'error')
total_levels = sum(r.get('max_level', 0) for r in results)
avg_actions = sum(r['actions'] for r in results) / max(len(results), 1)
players_found = sum(1 for r in results if r.get('player_found', False))

print('=' * 60)
print('SUBMISSION SUMMARY')
print('=' * 60)
print(f'  Games played:    {len(results)}/{len(game_ids)}')
print(f'  Wins:            {wins}')
print(f'  Errors:          {errors}')
print(f'  Total levels:    {total_levels}')
print(f'  Avg actions:     {avg_actions:.1f}')
print(f'  Players found:   {players_found}/{len(results)}')
print(f'  Total time:      {total_time:.1f}s')
print('=' * 60)

In [ ]:
# Save results for analysis
output_path = 'submission_results.json'
with open(output_path, 'w') as f:
    json.dump({
        'agent': 'AdaptiveReasoning',
        'version': 'v4_1',
        'max_actions': MAX_ACTIONS_PER_GAME,
        'total_time': total_time,
        'games_played': len(results),
        'wins': wins,
        'total_levels': total_levels,
        'results': results,
    }, f, indent=2, default=str)

print(f'Results saved to {output_path}')